# 🏭 Seller Artifact Enrichment Pipeline
**Purpose:** Transform semi-structured scraped seller artifact data (Excel/CSV) into structured seller enrichment outputs.  
**Outputs:** `company_details_output.xlsx`, `product_catalogue_output.xlsx`, `seller_offerings_output.xlsx` + debug files.  
**Platform:** Google Colab (free tier) — all tools are free/open-source.

---
## Architecture Overview
```
Input Excel/CSV
    │
    ▼
[1] Schema Inference        → detect semantic column roles (name, desc, url, sku, …)
    │
    ▼
[2] Cleaning & Normalization → nulls, whitespace, URLs, phones, emails
    │
    ▼
[3] Artifact Classification  → PRODUCT_CANDIDATE | COMPANY_INFO | MEDIA | BROCHURE | …
    │
    ├──[Hook 1: OCR]         → extract text from images/PDFs (if enabled)
    ├──[Hook 2: Image Cls]   → is image actually a product image? (if enabled)
    └──[Hook 3: LLM Extract] → structured extraction from messy text (if enabled, selective)
    │
    ▼
[4] Company Extraction       → field-level source-ranked company details
    │
    ▼
[5] Product Validation       → multi-signal scoring + hard exclusion layer
    │
    ├──[Hook 4: Spec Fetch]  → fetch missing specs from product page (if enabled)
    │
    ▼
[6] Product Extraction       → PRODUCT_CATALOGUE (real products only)
    │
    ▼
[7] Seller Offerings         → non-product commercially useful offerings
    │
    ▼
[8] Deduplication            → normalized name similarity grouping
    │
    ▼
[9] Export Outputs           → primary + debug files
    └──[Hook 5: Evaluation]  → precision/recall vs ground truth (if labels provided)
```


## Cell 1 — Install Dependencies

In [ ]:
# Core data & NLP
!pip install -q openpyxl rapidfuzz phonenumbers

# OCR (Hook 1) — EasyOCR: practical, GPU+CPU, multi-language
# !pip install -q easyocr  # Uncomment when ENABLE_OCR_HOOK = True

# Image classification (Hook 2) — CLIP via open_clip: lightweight, CPU-compatible
# !pip install -q open-clip-torch Pillow requests  # Uncomment when ENABLE_IMAGE_CLASSIFIER_HOOK = True

# LLM extraction (Hook 3) — transformers with a small instruct model
# !pip install -q transformers accelerate bitsandbytes  # Uncomment when ENABLE_LLM_EXTRACTION_HOOK = True

# Spec fetch (Hook 4) — already covered by stdlib + beautifulsoup
!pip install -q beautifulsoup4 lxml

# Evaluation (Hook 5)
# sklearn already available in Colab

print("✅ Core dependencies installed.")


## Cell 2 — Imports & Logging

In [ ]:
import os, re, json, time, logging, hashlib, warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process as rfprocess
import phonenumbers

warnings.filterwarnings("ignore")

# ── Logging setup ─────────────────────────────────────────────────────────────
LOG_LEVEL = logging.INFO
logging.basicConfig(
    level=LOG_LEVEL,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("SellerPipeline")
log.info("Pipeline initialised.")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("pipeline_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
log.info(f"Output directory: {OUTPUT_DIR}")


## Cell 3 — Configuration Flags
Tune thresholds and enable/disable hooks here. The pipeline runs fully even with all hooks disabled.

In [ ]:
# ── Hook enable/disable flags ─────────────────────────────────────────────────
ENABLE_OCR_HOOK               = False   # Hook 1: OCR text from images/PDFs
ENABLE_IMAGE_CLASSIFIER_HOOK  = False   # Hook 2: CLIP-based product image classifier
ENABLE_LLM_EXTRACTION_HOOK    = False   # Hook 3: LLM structured extraction (selective)
ENABLE_SPEC_FETCH_HOOK        = False   # Hook 4: Fetch specs from product page URLs
ENABLE_EVALUATION_HOOK        = False   # Hook 5: Precision/recall vs ground truth
GROUND_TRUTH_FILE             = None    # Path to ground-truth xlsx if available

# ── Pipeline thresholds ────────────────────────────────────────────────────────
PRODUCT_CONFIDENCE_THRESHOLD  = 0.40   # Min score for ACTUAL_PRODUCT inclusion
DEDUP_SIMILARITY_THRESHOLD    = 88     # RapidFuzz score (0-100) for deduplication
MAX_SPEC_FETCH_PER_RUN        = 20     # Safety cap on spec-fetch calls per run
LLM_MIN_CONFIDENCE_TO_TRIGGER = 0.35  # Only invoke LLM if rule-based conf <= this
LLM_MAX_ROWS_PER_RUN          = 30    # Safety cap on LLM hook calls per run
OCR_MIN_TEXT_LENGTH           = 30    # Minimum extracted text length to be useful
REQUEST_TIMEOUT               = 10    # Seconds for HTTP requests (spec fetch)

# ── Column mapping hints (extend as needed) ───────────────────────────────────
# Fuzzy keyword groups per semantic role
SEMANTIC_ROLE_KEYWORDS = {
    "seller_id":        ["unique identifier", "seller id", "seller_id", "uid", "id"],
    "source_url":       ["source url", "website", "domain", "homepage", "source"],
    "found_on_page":    ["found on page", "page url", "found on", "scraped from"],
    "item_type":        ["item type", "type", "content type", "artifact type"],
    "item_name":        ["name", "title", "product name", "item name", "heading"],
    "brand":            ["brand", "brand name", "manufacturer", "maker"],
    "sku":              ["sku", "model", "part number", "model number", "item code", "product code"],
    "artifact_link":    ["link", "url", "href", "product url", "item url", "page link"],
    "description":      ["description", "desc", "details", "about", "summary", "content", "text"],
    "price":            ["price", "cost", "rate", "mrp", "amount"],
    "product_specifications": ["product specifications", "specifications", "specs", "technical", "spec"],
    "ratings":          ["ratings", "rating", "stars", "score"],
    "reviews":          ["reviews", "review", "feedback"],
    "other_info":       ["any other information", "other info", "additional", "notes", "remarks"],
    "image_url":        ["image url", "image", "photo url", "img url", "thumbnail"],
    "video_url":        ["video url", "video", "youtube", "vimeo"],
    "document_url":     ["document url", "document", "pdf url", "brochure url", "catalogue url"],
    "category_format":  ["category/format", "category", "format", "section", "type label"],
    "detection_method": ["detection method", "method", "scrape method", "found by"],
    "raw_text":         ["raw text", "page text", "full text", "scraped text"],
    "page_title":       ["page title", "title tag", "meta title"],
}

# ── Hard exclusion patterns (used in product validation) ──────────────────────
HARD_EXCLUSION_PATTERNS = [
    r"^(welcome|about us|our story|vision|mission|why us|why choose|our philosophy|our approach)[\s\W]",
    r"^(contact|get in touch|reach us|send us|connect with|talk to|drop us)",
    r"^(opening hours?|business hours?|working hours?|timing)",
    r"^\d+[\+\-]?\s*(years?|clients?|projects?|sq\.?\s*ft\.?|happy|customers?)",
    r"^(professional|diverse|expert|result|trusted|proven|certified)\s+(team|approach|advice|based|partner)",
    r"^(save your|professional supplier|leading developer|crafting exceptional)",
    r"^(location|address|map|direction|find us|where are we)",
    r"^(faqs?|terms|conditions|privacy policy|disclaimer|legal|cookies)",
    r"^(follow us|social media|connect on|facebook|instagram|linkedin|twitter)",
    r"^(download|subscribe|sign up|get quote|request|enquiry|inquiry|brochure)\s*$",
    r"^(our product|our service|our solution|our offer)s?\s*$",
    r"^(introduction|overview|infrastructure|track record|proven track)",
    r"^(industries?\s+served|sectors?\s+served|our client|happy client|client testimonial)",
    r"(absolute security|peace of mind|your dream|luxury lifestyle|premium living)",
    r"^(discover|explore|learn more|read more|know more|view more|see more)\s*$",
    r"^(filing complaints?|general terms|terms of use|use of (the )?website)",
    r"solve the below|captcha|what is \d",
    r"^(marketing office|head office|registered office|corporate office)\s*$",
    r"^(oh&s|safety provision|protective equipment|cardinal rules?)",
]

# ── Category-level name patterns (not individual products) ────────────────────
CATEGORY_PATTERNS = [
    r"^(our\s+)?(products?|services?|solutions?|offerings?|collections?|categories|range|portfolio)\s*$",
    r"^(industrial|commercial|residential|premium|luxury|standard)\s+(pumps?|valves?|tanks?|pipes?)$",
    r"^(concrete\s+additives?|injection\s+grouts?|woodgrain|solid\s+colour|stone|marble)$",
    r"^(desserts?|appetizers?|starters?|mains?|sides?|beverages?|drinks?)\s*$",
    r"^(chicken|mutton|veg|non-?veg|seafood)\s+(dishes?|items?|section)\s*$",
    r"^(demat\s+account|equity|mutual\s+fund|insurance|fixed\s+deposit)\s*$",
]

# ── Company page trust sources (higher index = lower trust) ───────────────────
COMPANY_SOURCE_TRUST = [
    "about", "contact", "footer", "homepage",
    "brochure", "product_page", "generic_page", "unknown"
]

log.info("✅ Configuration loaded.")


## Cell 4 — Utility Helpers

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# General utilities
# ─────────────────────────────────────────────────────────────────────────────

def safe_str(val) -> str:
    """Convert any value to a stripped string, or empty string if null."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return ""
    return str(val).strip()


def normalize_url(url: str) -> str:
    """Basic URL normalization: strip trailing slashes, lowercase scheme+host."""
    url = safe_str(url)
    if not url:
        return ""
    url = url.strip().rstrip("/")
    # Ensure scheme
    if url and not url.startswith(("http://", "https://")):
        url = "https://" + url
    return url


def extract_emails(text: str) -> list:
    """Extract email addresses from arbitrary text."""
    if not text:
        return []
    return list(set(re.findall(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}", text)))


def extract_phones(text: str) -> list:
    """Extract phone numbers from text using phonenumbers + regex fallback."""
    if not text:
        return []
    found = set()
    # phonenumbers library pass
    for match in phonenumbers.PhoneNumberMatcher(text, "IN"):
        found.add(phonenumbers.format_number(match.number, phonenumbers.PhoneNumberFormat.E164))
    # Regex fallback for obvious patterns missed by library
    raw = re.findall(r"[+]?[\d][\d\s\-().]{7,}[\d]", text)
    for r in raw:
        cleaned = re.sub(r"[\s\-().]", "", r)
        if 9 <= len(cleaned) <= 15:
            found.add(cleaned)
    return sorted(found)


def normalize_text(text: str, max_len: int = 2000) -> str:
    """Normalize whitespace and truncate."""
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len]


def row_text_blob(row: pd.Series, cols: list) -> str:
    """Concatenate multiple columns into one text blob for NLP."""
    parts = [safe_str(row.get(c, "")) for c in cols if safe_str(row.get(c, ""))]
    return " ".join(parts)


def compute_row_hash(row: pd.Series) -> str:
    """Create a short hash from the full row string for dedup tracking."""
    return hashlib.md5(str(row.values).encode()).hexdigest()[:10]


def fuzzy_col_match(col_name: str, keyword_list: list, threshold: int = 75) -> bool:
    """Return True if col_name fuzzy-matches any keyword in keyword_list."""
    col_lower = col_name.lower()
    for kw in keyword_list:
        if fuzz.partial_ratio(col_lower, kw) >= threshold:
            return True
    return False


def url_path_tokens(url: str) -> list:
    """Extract meaningful path tokens from a URL."""
    url = safe_str(url)
    path = re.sub(r"https?://[^/]+", "", url)
    tokens = re.split(r"[/\-_?=&.]+", path.lower())
    return [t for t in tokens if len(t) > 2]


def is_likely_product_url(url: str) -> bool:
    """Heuristic: does the URL path suggest a specific product page?"""
    tokens = url_path_tokens(url)
    product_signals = {"product", "item", "shop", "catalogue", "sku", "buy", "detail", "listing", "pd", "p"}
    return bool(product_signals & set(tokens))


def is_likely_company_url(url: str) -> bool:
    """Heuristic: does the URL path suggest an about/contact/home page?"""
    tokens = url_path_tokens(url)
    company_signals = {"about", "contact", "home", "team", "us", "company", "who", "profile", "info", "footer"}
    return bool(company_signals & set(tokens))


def matches_any_pattern(text: str, patterns: list) -> bool:
    """Return True if text matches any compiled regex pattern."""
    t = text.strip().lower()
    for pat in patterns:
        if re.search(pat, t, re.IGNORECASE):
            return True
    return False


def classify_page_source(url: str) -> str:
    """Classify a URL into a company source trust tier."""
    url_l = safe_str(url).lower()
    if any(k in url_l for k in ["about", "our-story", "who-we-are"]):
        return "about"
    if any(k in url_l for k in ["contact", "reach-us", "get-in-touch"]):
        return "contact"
    if any(k in url_l for k in ["footer", "home", "index"]) or url_l.endswith(".com") or url_l.endswith(".in"):
        return "homepage"
    if any(k in url_l for k in ["product", "shop", "item", "catalogue", "buy"]):
        return "product_page"
    if any(k in url_l for k in ["brochure", "pdf", "download"]):
        return "brochure"
    return "generic_page"


def source_trust_rank(source: str) -> int:
    """Lower integer = higher trust."""
    try:
        return COMPANY_SOURCE_TRUST.index(source)
    except ValueError:
        return len(COMPANY_SOURCE_TRUST)


def safe_excel_write(df: pd.DataFrame, path: str, sheet_name: str = "Sheet1") -> None:
    """Write DataFrame to Excel with basic formatting, safely."""
    try:
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            df.to_excel(writer, index=False, sheet_name=sheet_name)
            ws = writer.sheets[sheet_name]
            # Auto-width (cap at 60)
            for col_cells in ws.columns:
                max_len = max((len(safe_str(c.value)) for c in col_cells), default=10)
                ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 2, 60)
        log.info(f"  Saved → {path}  ({len(df)} rows)")
    except Exception as e:
        log.error(f"  Failed to write {path}: {e}")


log.info("✅ Utility helpers loaded.")


## Cell 5 — Schema Inference / Column Role Detection

In [ ]:
def infer_column_roles(df: pd.DataFrame) -> dict:
    """
    Map actual DataFrame columns to semantic roles using keyword + fuzzy matching.
    Returns dict: role -> [list of matching column names]
    Multi-column roles are supported (e.g., two description columns).
    """
    role_map = defaultdict(list)
    used_cols = set()

    for role, keywords in SEMANTIC_ROLE_KEYWORDS.items():
        for col in df.columns:
            if col in used_cols:
                continue
            if fuzzy_col_match(col, keywords, threshold=70):
                role_map[role].append(col)
                # Only mark as 'used' for single-value roles to avoid double-mapping
                if role in ("seller_id", "source_url", "item_type", "sku", "price"):
                    used_cols.add(col)

    log.info("Inferred column roles:")
    for role, cols in role_map.items():
        log.info(f"  {role:30s} → {cols}")
    return dict(role_map)


def get_first(row: pd.Series, cols: list, default="") -> str:
    """Return the first non-empty value from a list of columns."""
    for col in cols:
        val = safe_str(row.get(col, ""))
        if val:
            return val
    return default


def build_unified_row(row: pd.Series, role_map: dict) -> dict:
    """
    Build a standardized unified dict from a raw row using the role map.
    This decouples downstream logic from column names.
    """
    def g(role):
        return get_first(row, role_map.get(role, []))
    def g_all(role):
        parts = []
        for col in role_map.get(role, []):
            v = safe_str(row.get(col, ""))
            if v:
                parts.append(v)
        return " | ".join(parts)

    return {
        "seller_id":          g("seller_id"),
        "source_url":         normalize_url(g("source_url")),
        "found_on_page":      normalize_url(g("found_on_page")),
        "item_type":          g("item_type"),
        "item_name":          g_all("item_name"),
        "brand":              g("brand"),
        "sku":                g("sku"),
        "artifact_link":      normalize_url(g("artifact_link")),
        "description":        g_all("description"),
        "price":              g("price"),
        "product_specifications": g_all("product_specifications"),
        "ratings":            g("ratings"),
        "reviews":            g("reviews"),
        "other_info":         g_all("other_info"),
        "image_url":          normalize_url(g("image_url")),
        "video_url":          normalize_url(g("video_url")),
        "document_url":       normalize_url(g("document_url")),
        "category_format":    g("category_format"),
        "detection_method":   g("detection_method"),
        "raw_text":           g_all("raw_text"),
        "page_title":         g("page_title"),
    }


log.info("✅ Schema inference functions loaded.")


## Cell 6 — Data Ingestion

In [ ]:
def load_input_file(file_path: str) -> pd.DataFrame:
    """
    Load Excel (single or multi-sheet) or CSV into one merged DataFrame.
    Preserves source sheet name and original row index for traceability.
    """
    path = Path(file_path)
    log.info(f"Loading: {path.name}")
    frames = []

    if path.suffix.lower() in (".xlsx", ".xls", ".xlsm"):
        all_sheets = pd.read_excel(path, sheet_name=None, dtype=str)
        for sheet_name, df in all_sheets.items():
            df = df.copy()
            df["_source_sheet"] = sheet_name
            df["_original_row_idx"] = range(len(df))
            frames.append(df)
            log.info(f"  Sheet '{sheet_name}': {df.shape}")
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, dtype=str)
        df["_source_sheet"] = "Sheet1"
        df["_original_row_idx"] = range(len(df))
        frames.append(df)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    merged = pd.concat(frames, ignore_index=True)
    # Normalize NaN-like strings to actual NaN
    merged.replace(["nan", "NaN", "None", "NULL", "N/A", "n/a", "#N/A", ""], np.nan, inplace=True)
    log.info(f"Total rows loaded: {len(merged)} | Columns: {list(merged.columns)}")
    return merged


log.info("✅ Ingestion function loaded.")


## Cell 7 — Cleaning & Normalization

In [ ]:
def clean_and_normalize(raw_df: pd.DataFrame, role_map: dict) -> pd.DataFrame:
    """
    Clean raw DataFrame and build a unified normalized DataFrame.
    Returns a new DataFrame with standardized columns + original raw columns preserved.
    """
    unified_rows = []
    for idx, row in raw_df.iterrows():
        urow = build_unified_row(row, role_map)
        urow["_raw_row_idx"]    = idx
        urow["_source_sheet"]   = safe_str(row.get("_source_sheet", ""))
        urow["_row_hash"]       = compute_row_hash(row)
        # Preserve raw columns for traceability
        for col in raw_df.columns:
            urow[f"_raw_{col}"] = safe_str(row.get(col, ""))
        unified_rows.append(urow)

    df = pd.DataFrame(unified_rows)

    # ── Deduplicate exact duplicate rows (same hash) ──────────────────────────
    before = len(df)
    df = df.drop_duplicates(subset=["_row_hash"]).reset_index(drop=True)
    log.info(f"Exact dedup: {before} → {len(df)} rows")

    # ── Build composite text blob for each row ────────────────────────────────
    text_cols = ["item_name", "description", "product_specifications", "other_info", "raw_text", "page_title"]
    df["_text_blob"] = df.apply(lambda r: normalize_text(
        " ".join([safe_str(r.get(c, "")) for c in text_cols if safe_str(r.get(c, ""))])
    ), axis=1)

    # ── Extract emails & phones from text blob ─────────────────────────────────
    df["_emails_found"]  = df["_text_blob"].apply(extract_emails)
    df["_phones_found"]  = df["_text_blob"].apply(extract_phones)

    # ── Page source classification ─────────────────────────────────────────────
    df["_page_source_type"] = df["found_on_page"].apply(classify_page_source)

    log.info(f"Cleaned dataset: {df.shape}")
    return df


log.info("✅ Cleaning functions loaded.")


## Cell 8 — Artifact Classification

In [ ]:
ARTIFACT_CLASSES = [
    "ACTUAL_PRODUCT_CANDIDATE",
    "PRODUCT_SUPPORTING_MEDIA",
    "COMPANY_INFO_EVIDENCE",
    "DOCUMENT_OR_BROCHURE",
    "GENERAL_WEBPAGE",
    "IRRELEVANT_MEDIA",
    "UNKNOWN",
]

COMPANY_INFO_KEYWORDS = [
    "about", "contact", "address", "email", "phone", "our story", "who we are",
    "opening hours", "business hours", "location", "directions", "get in touch",
    "company profile", "welcome to", "established", "founded", "year",
]

PRODUCT_EVIDENCE_KEYWORDS = [
    "sku", "model", "specifications", "spec", "buy", "price", "add to cart",
    "in stock", "available", "material", "dimension", "weight", "capacity",
    "warranty", "product details", "technical details",
]

MEDIA_IRRELEVANT_KEYWORDS = [
    "logo", "banner", "favicon", "icon", "background", "hero", "slider",
    "advertisement", "ad", "promo", "decorative",
]


def classify_artifact(row: pd.Series) -> str:
    """
    Rule-based artifact classification.
    Returns one of ARTIFACT_CLASSES.
    """
    item_type       = safe_str(row.get("item_type", "")).lower()
    cat_format      = safe_str(row.get("category_format", "")).lower()
    name            = safe_str(row.get("item_name", "")).lower()
    desc            = safe_str(row.get("description", "")).lower()
    doc_url         = safe_str(row.get("document_url", ""))
    image_url       = safe_str(row.get("image_url", ""))
    video_url       = safe_str(row.get("video_url", ""))
    found_on        = safe_str(row.get("found_on_page", "")).lower()
    sku             = safe_str(row.get("sku", ""))
    specs           = safe_str(row.get("product_specifications", ""))
    text_blob       = safe_str(row.get("_text_blob", "")).lower()

    # ── Documents / brochures ─────────────────────────────────────────────────
    if "catalog" in item_type or "document" in item_type or doc_url:
        if any(k in (name + cat_format) for k in ["brochure", "catalogue", "catalog", "pdf", "download"]):
            return "DOCUMENT_OR_BROCHURE"

    # ── Pure image/video media ────────────────────────────────────────────────
    if "image" in item_type or "media/image" in cat_format:
        if any(k in image_url.lower() for k in MEDIA_IRRELEVANT_KEYWORDS):
            return "IRRELEVANT_MEDIA"
        # Image could still support product
        if sku or specs or any(k in text_blob for k in PRODUCT_EVIDENCE_KEYWORDS):
            return "PRODUCT_SUPPORTING_MEDIA"
        return "PRODUCT_SUPPORTING_MEDIA"

    if "video" in item_type or video_url:
        return "PRODUCT_SUPPORTING_MEDIA"

    # ── Company info evidence ─────────────────────────────────────────────────
    if any(k in text_blob for k in COMPANY_INFO_KEYWORDS):
        company_score = sum(1 for k in COMPANY_INFO_KEYWORDS if k in text_blob)
        product_score = sum(1 for k in PRODUCT_EVIDENCE_KEYWORDS if k in text_blob)
        if company_score > product_score + 1:
            return "COMPANY_INFO_EVIDENCE"

    # ── Product candidate ─────────────────────────────────────────────────────
    if "product" in item_type or "webpage" in cat_format.lower():
        return "ACTUAL_PRODUCT_CANDIDATE"

    return "UNKNOWN"


def add_artifact_classification(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["artifact_class"] = df.apply(classify_artifact, axis=1)
    log.info("Artifact class distribution:")
    for cls, cnt in df["artifact_class"].value_counts().items():
        log.info(f"  {cls}: {cnt}")
    return df


log.info("✅ Artifact classification loaded.")


## Cell 9 — AI / ML Extension Hooks
All hooks are **integrated into the pipeline flow** but **disabled by default**.  
Each hook falls back gracefully if disabled or if it fails.

| Hook | Tool | Why |
|------|------|-----|
| OCR | EasyOCR | Best free multi-lang OCR, GPU+CPU |
| Image Classifier | OpenCLIP (ViT-B-32) | Zero-shot product vs non-product, ~300MB |
| LLM Extraction | `flan-t5-base` via 🤗 transformers | 250MB, CPU-friendly seq2seq, free |
| Spec Fetch | requests + BeautifulSoup | Lightweight, no GPU needed |
| Evaluation | sklearn | Standard metrics |


In [ ]:
# ── Hook 1: OCR ───────────────────────────────────────────────────────────────

_ocr_reader = None

def _get_ocr_reader():
    global _ocr_reader
    if _ocr_reader is None:
        import easyocr
        _ocr_reader = easyocr.Reader(["en"], gpu=False, verbose=False)
    return _ocr_reader


def hook_ocr_extraction(url: str) -> dict:
    """
    Hook 1: Extract text from an image or PDF URL.
    Strategy:
      1. For PDFs: try pdfminer direct text extraction first.
      2. If weak/empty, and ENABLE_OCR_HOOK: run EasyOCR on rendered page images.
      3. For images: run EasyOCR directly.
    Returns: {"text": str, "method": str, "confidence": float}
    """
    if not url:
        return {"text": "", "method": "none", "confidence": 0.0}

    result = {"text": "", "method": "none", "confidence": 0.0}
    url_lower = url.lower()

    try:
        import requests
        from io import BytesIO

        # ── PDF: try direct text extraction first ─────────────────────────────
        if url_lower.endswith(".pdf") or "pdf" in url_lower:
            try:
                from pdfminer.high_level import extract_text
                resp = requests.get(url, timeout=REQUEST_TIMEOUT, headers={"User-Agent": "Mozilla/5.0"})
                text = extract_text(BytesIO(resp.content))
                text = normalize_text(text)
                if len(text) >= OCR_MIN_TEXT_LENGTH:
                    result = {"text": text[:3000], "method": "pdf_text_direct", "confidence": 0.85}
                    return result
            except Exception as e:
                log.debug(f"PDF direct extract failed ({url}): {e}")

        # ── Image OCR (or PDF fallback) ───────────────────────────────────────
        if ENABLE_OCR_HOOK:
            reader = _get_ocr_reader()
            resp = requests.get(url, timeout=REQUEST_TIMEOUT, headers={"User-Agent": "Mozilla/5.0"})
            from PIL import Image
            img = Image.open(BytesIO(resp.content)).convert("RGB")
            ocr_result = reader.readtext(img, detail=1, paragraph=True)
            text = " ".join([item[1] for item in ocr_result])
            conf = float(np.mean([item[2] for item in ocr_result])) if ocr_result else 0.0
            if len(text) >= OCR_MIN_TEXT_LENGTH:
                result = {"text": text[:3000], "method": "easyocr", "confidence": conf}
            else:
                result = {"text": text, "method": "easyocr_weak", "confidence": conf}
        else:
            result = {"text": "", "method": "ocr_disabled", "confidence": 0.0}

    except Exception as e:
        log.warning(f"Hook 1 (OCR) failed for {url}: {e}")
        result = {"text": "", "method": "ocr_error", "confidence": 0.0}

    return result


# ── Hook 2: Image Product Classifier ─────────────────────────────────────────

_clip_model = None
_clip_preprocess = None
_clip_tokenizer = None

def _get_clip():
    global _clip_model, _clip_preprocess, _clip_tokenizer
    if _clip_model is None:
        import open_clip
        _clip_model, _, _clip_preprocess = open_clip.create_model_and_transforms(
            "ViT-B-32", pretrained="openai"
        )
        _clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
        _clip_model.eval()
    return _clip_model, _clip_preprocess, _clip_tokenizer


CLIP_PRODUCT_PROMPTS = [
    "a photo of a product for sale",
    "a product image on an e-commerce website",
    "a photo of merchandise",
]
CLIP_NONPRODUCT_PROMPTS = [
    "a company logo or banner",
    "a marketing background image",
    "a decorative photo with no product",
    "a stock photo of people or scenery",
]


def hook_image_product_classifier(image_url: str) -> dict:
    """
    Hook 2: Zero-shot product image classifier using OpenCLIP (ViT-B-32).
    Returns: {"is_product_image": bool, "confidence": float, "category": str}
    IMPORTANT: Acts only as secondary confidence modifier, not primary decider.
    """
    if not image_url or not ENABLE_IMAGE_CLASSIFIER_HOOK:
        return {"is_product_image": None, "confidence": 0.0, "category": "unknown", "skipped": True}
    try:
        import torch, requests
        from PIL import Image as PILImage
        from io import BytesIO
        import open_clip

        model, preprocess, tokenizer = _get_clip()
        resp = requests.get(image_url, timeout=REQUEST_TIMEOUT, headers={"User-Agent": "Mozilla/5.0"})
        img = PILImage.open(BytesIO(resp.content)).convert("RGB")
        img_tensor = preprocess(img).unsqueeze(0)

        all_prompts = CLIP_PRODUCT_PROMPTS + CLIP_NONPRODUCT_PROMPTS
        text_tokens = tokenizer(all_prompts)
        with torch.no_grad():
            img_features  = model.encode_image(img_tensor)
            text_features = model.encode_text(text_tokens)
            img_features  /= img_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            sims = (img_features @ text_features.T).squeeze(0).softmax(dim=-1).numpy()

        product_score    = float(sims[:len(CLIP_PRODUCT_PROMPTS)].sum())
        nonproduct_score = float(sims[len(CLIP_PRODUCT_PROMPTS):].sum())
        is_product       = product_score > nonproduct_score
        confidence       = product_score if is_product else nonproduct_score

        return {
            "is_product_image": is_product,
            "confidence": round(confidence, 3),
            "category": "product" if is_product else "non_product",
            "skipped": False,
        }
    except Exception as e:
        log.warning(f"Hook 2 (Image Classifier) failed for {image_url}: {e}")
        return {"is_product_image": None, "confidence": 0.0, "category": "error", "skipped": True}


# ── Hook 3: LLM Extraction ───────────────────────────────────────────────────

_llm_pipeline = None

def _get_llm_pipeline():
    """Lazy-load flan-t5-base (250MB, CPU-friendly seq2seq instruct model)."""
    global _llm_pipeline
    if _llm_pipeline is None:
        from transformers import pipeline
        log.info("Loading flan-t5-base for LLM hook (first time ~1min)...")
        _llm_pipeline = pipeline(
            "text2text-generation",
            model="google/flan-t5-base",
            max_new_tokens=200,
        )
        log.info("flan-t5-base loaded.")
    return _llm_pipeline


LLM_PROMPTS = {
    "product": (
        "Extract structured product info from this text. "
        "Output JSON with keys: product_name, category, key_features, materials, specifications. "
        "If a field is unknown, use null. "
        "Text: {text}"
    ),
    "company": (
        "Extract company info from this text. "
        "Output JSON with keys: company_name, industry, description, city, country. "
        "If a field is unknown, use null. "
        "Text: {text}"
    ),
}


def hook_llm_extraction(text: str, task: str = "product") -> dict:
    """
    Hook 3: Selective LLM structured extraction using flan-t5-base.
    Only invoked for low-confidence rows. Never on every row.
    Model: google/flan-t5-base — 250MB, free, CPU-friendly.
    Limitations vs GPT-4: weaker reasoning, may produce partial JSON.
    Returns: dict with extracted fields or {"error": ...}.
    """
    if not ENABLE_LLM_EXTRACTION_HOOK or not text or len(text.strip()) < 30:
        return {"skipped": True, "reason": "disabled or empty"}
    try:
        pipe = _get_llm_pipeline()
        prompt_template = LLM_PROMPTS.get(task, LLM_PROMPTS["product"])
        prompt = prompt_template.format(text=text[:800])
        out = pipe(prompt, max_new_tokens=200)[0]["generated_text"]
        # Try to parse JSON; flan-t5 sometimes returns partial JSON
        out_clean = out.strip()
        # Strip markdown code fences if present
        out_clean = re.sub(r"```json|```", "", out_clean).strip()
        try:
            parsed = json.loads(out_clean)
        except json.JSONDecodeError:
            # Try to extract partial JSON
            match = re.search(r"\{.*\}", out_clean, re.DOTALL)
            parsed = json.loads(match.group()) if match else {"raw_output": out_clean}
        parsed["skipped"] = False
        return parsed
    except Exception as e:
        log.warning(f"Hook 3 (LLM) failed: {e}")
        return {"error": str(e), "skipped": True}


# ── Hook 4: Spec Fetch ────────────────────────────────────────────────────────

def hook_fetch_product_specs(product_page_url: str) -> dict:
    """
    Hook 4: Fetch a product page URL and extract spec tables / spec-like sections.
    Uses requests + BeautifulSoup with timeouts and error handling.
    Returns: {"specs_text": str, "method": str}
    """
    if not product_page_url or not ENABLE_SPEC_FETCH_HOOK:
        return {"specs_text": "", "method": "disabled"}
    try:
        import requests
        from bs4 import BeautifulSoup

        resp = requests.get(
            product_page_url,
            timeout=REQUEST_TIMEOUT,
            headers={"User-Agent": "Mozilla/5.0 (compatible; SellerPipeline/1.0)"},
        )
        if resp.status_code != 200:
            return {"specs_text": "", "method": f"http_{resp.status_code}"}

        soup = BeautifulSoup(resp.text, "lxml")

        # Remove nav/footer/script noise
        for tag in soup(["nav", "footer", "script", "style", "header", "aside"]):
            tag.decompose()

        extracted = []

        # Try: <table> elements that look like spec tables
        for table in soup.find_all("table"):
            rows_text = []
            for tr in table.find_all("tr"):
                cells = [td.get_text(separator=" ", strip=True) for td in tr.find_all(["td", "th"])]
                if len(cells) >= 2:
                    rows_text.append(" | ".join(cells))
            if rows_text:
                extracted.append("\n".join(rows_text))

        # Try: dl/dt/dd spec lists
        for dl in soup.find_all("dl"):
            pairs = []
            terms = dl.find_all("dt")
            defs  = dl.find_all("dd")
            for dt, dd in zip(terms, defs):
                pairs.append(f"{dt.get_text(strip=True)}: {dd.get_text(strip=True)}")
            if pairs:
                extracted.append("\n".join(pairs))

        # Try: sections with "spec" in class/id
        for el in soup.find_all(attrs={"class": re.compile(r"spec|feature|detail|attribute", re.I)}):
            t = el.get_text(separator="\n", strip=True)
            if len(t) > 30:
                extracted.append(t[:500])

        specs_text = normalize_text("\n\n".join(extracted))
        return {
            "specs_text": specs_text[:2000] if specs_text else "",
            "method": "bs4_table_dl_class",
        }
    except Exception as e:
        log.warning(f"Hook 4 (Spec Fetch) failed for {product_page_url}: {e}")
        return {"specs_text": "", "method": f"error: {e}"}


# ── Hook 5: Evaluation ────────────────────────────────────────────────────────

def hook_evaluate_precision_recall(
    predicted_df: pd.DataFrame,
    ground_truth_df: pd.DataFrame,
    pred_col: str = "is_actual_product",
    gt_col: str   = "is_actual_product",
    join_col: str = "_raw_row_idx",
) -> dict:
    """
    Hook 5: Compare pipeline predictions against ground truth labels.
    Ground truth file must have a join column (default: _raw_row_idx) and
    a boolean/0-1 column for the label.
    Returns precision, recall, F1, and confusion matrix.
    """
    if not ENABLE_EVALUATION_HOOK or ground_truth_df is None:
        return {"skipped": True, "reason": "disabled or no ground truth"}
    try:
        from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

        merged = predicted_df[[join_col, pred_col]].merge(
            ground_truth_df[[join_col, gt_col]], on=join_col, suffixes=("_pred", "_gt")
        )
        y_pred = merged[f"{pred_col}_pred"].astype(int)
        y_true = merged[f"{gt_col}_gt"].astype(int)

        metrics = {
            "precision":        round(precision_score(y_true, y_pred, zero_division=0), 4),
            "recall":           round(recall_score(y_true, y_pred, zero_division=0), 4),
            "f1":               round(f1_score(y_true, y_pred, zero_division=0), 4),
            "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
            "n_evaluated":      len(merged),
            "skipped":          False,
        }
        log.info(f"Evaluation: {metrics}")
        return metrics
    except Exception as e:
        log.error(f"Hook 5 (Evaluation) failed: {e}")
        return {"error": str(e), "skipped": True}


log.info("✅ All hook functions loaded.")


## Cell 10 — Optional Artifact Enrichment
This is the hook integration point in the pipeline. Runs OCR, image classification,
and spec-fetch enrichment on candidate rows before product validation.

In [ ]:
def enrich_artifacts(df: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    """
    Run enabled hooks on rows that would benefit from enrichment.
    Returns enriched DataFrame and a hook debug log.
    """
    df = df.copy()
    df["_ocr_text"]           = ""
    df["_ocr_method"]         = ""
    df["_image_cls_result"]   = ""
    df["_image_cls_conf"]     = 0.0
    df["_llm_extracted"]      = ""
    df["_hook_triggered"]     = ""

    hook_log = []  # For hook_debug_log_output.xlsx

    # ── Candidate rows for enrichment ─────────────────────────────────────────
    ocr_candidates = df[
        (df["artifact_class"].isin(["ACTUAL_PRODUCT_CANDIDATE", "DOCUMENT_OR_BROCHURE", "PRODUCT_SUPPORTING_MEDIA"]))
        & (df["_text_blob"].str.len() < 100)
        & (df["document_url"].str.len() > 0 | df["image_url"].str.len() > 0)
    ].index.tolist()

    img_cls_candidates = df[
        (df["artifact_class"].isin(["ACTUAL_PRODUCT_CANDIDATE", "PRODUCT_SUPPORTING_MEDIA"]))
        & (df["image_url"].str.len() > 0)
    ].index.tolist()

    log.info(f"Enrichment candidates — OCR: {len(ocr_candidates)}, ImgCls: {len(img_cls_candidates)}")

    # ── Hook 1: OCR ───────────────────────────────────────────────────────────
    for idx in ocr_candidates:
        row = df.loc[idx]
        url = row.get("document_url") or row.get("image_url")
        result = hook_ocr_extraction(url)
        df.at[idx, "_ocr_text"]   = result["text"]
        df.at[idx, "_ocr_method"] = result["method"]
        if result["text"]:
            # Augment text blob with OCR text
            df.at[idx, "_text_blob"] = (df.at[idx, "_text_blob"] + " " + result["text"]).strip()
            triggered = df.at[idx, "_hook_triggered"]
            df.at[idx, "_hook_triggered"] = (triggered + "|OCR").strip("|")
        hook_log.append({
            "row_idx": idx, "hook": "OCR",
            "url": url, "result_preview": result["text"][:100],
            "method": result["method"], "confidence": result.get("confidence", 0),
        })

    # ── Hook 2: Image Classifier ──────────────────────────────────────────────
    for idx in img_cls_candidates:
        img_url = df.at[idx, "image_url"]
        result  = hook_image_product_classifier(img_url)
        df.at[idx, "_image_cls_result"] = str(result.get("category", ""))
        df.at[idx, "_image_cls_conf"]   = result.get("confidence", 0.0)
        if not result.get("skipped"):
            triggered = df.at[idx, "_hook_triggered"]
            df.at[idx, "_hook_triggered"] = (triggered + "|IMG_CLS").strip("|")
        hook_log.append({
            "row_idx": idx, "hook": "IMAGE_CLS",
            "url": img_url, "result_preview": str(result),
            "method": "CLIP", "confidence": result.get("confidence", 0),
        })

    log.info("✅ Artifact enrichment complete.")
    return df, hook_log


log.info("✅ Artifact enrichment loaded.")


## Cell 11 — Company Details Extraction

In [ ]:
CERT_PATTERNS = [
    r"\bISO\s*\d{4,5}(?:[:\-]\d{4})?\b",
    r"\bBIS\b",
    r"\bFSSAI\b",
    r"\bGST(?:IN)?\b",
    r"\bSEBI\s+Regn(?:\s+No\.?)?\s*[A-Z0-9\-]+",
    r"\bAMFI\b",
    r"\bCE\s+(?:mark(?:ed)?|certified|certification)\b",
    r"\bRoHS\b",
    r"\bIEC\s+\d+\b",
    r"\bUL(?:\s+listed)?\b",
    r"\bNSF\b",
    r"\bHACCP\b",
    r"\bOSHAS?\s*\d+\b",
    r"\bAS\s*\d{4,5}\b",
]

BUSINESS_TYPE_KEYWORDS = {
    "manufacturer": ["manufacturer", "manufacturing", "manufactures", "we manufacture", "produced by"],
    "distributor":  ["distributor", "distribution", "distributes", "authorized dealer"],
    "exporter":     ["exporter", "exports", "we export", "export house"],
    "importer":     ["importer", "imports", "we import"],
    "trader":       ["trader", "trading", "trade house"],
    "service":      ["services", "service provider", "consulting", "solutions"],
    "retailer":     ["retailer", "retail", "shop", "store"],
    "restaurant":   ["restaurant", "hotel", "food", "dining", "menu", "cuisine"],
    "real_estate":  ["developer", "properties", "apartments", "villas", "residential", "commercial spaces"],
    "financial":    ["securities", "broking", "mutual fund", "insurance", "demat", "trading account"],
}

YEAR_PATTERN = re.compile(r"\b(19[5-9]\d|20[0-2]\d)\b")


def extract_certifications(text: str) -> list:
    """Safe, boundary-aware certification extraction."""
    certs = set()
    for pat in CERT_PATTERNS:
        for m in re.finditer(pat, text, re.IGNORECASE):
            certs.add(m.group().strip())
    return sorted(certs)


def extract_year_established(text: str) -> str:
    matches = YEAR_PATTERN.findall(text)
    if matches:
        # Return earliest plausible year
        return str(min(int(y) for y in matches))
    return ""


def infer_business_type_flags(text: str) -> dict:
    t_lower = text.lower()
    flags = {}
    for btype, keywords in BUSINESS_TYPE_KEYWORDS.items():
        flags[btype] = any(k in t_lower for k in keywords)
    return flags


def extract_company_for_seller(seller_rows: pd.DataFrame) -> dict:
    """
    Build a company record from rows belonging to one seller.
    Uses field-level source ranking: prefer about/contact/homepage rows.
    """
    company = {
        "seller_id": "", "registered_company_name": "", "brand_name": "",
        "website": "", "primary_phone": "", "alternate_phones": "",
        "primary_email": "", "alternate_emails": "", "whatsapp_number": "",
        "registered_address": "", "city": "", "state": "", "pincode": "", "country": "India",
        "business_type": "", "industry": "", "sub_industry": "",
        "company_description": "", "year_established": "", "years_in_business": "",
        "certifications": "", "exporter_flag": False, "importer_flag": False,
        "manufacturer_flag": False, "distributor_flag": False,
        "facebook_url": "", "instagram_url": "", "linkedin_url": "", "youtube_url": "",
        "company_data_confidence_score": 0.0, "company_data_sources": "",
        "missing_fields": "", "notes_for_manual_review": "",
    }

    seller_id = safe_str(seller_rows["seller_id"].iloc[0]) if "seller_id" in seller_rows.columns else ""
    company["seller_id"] = seller_id

    # ── Website ───────────────────────────────────────────────────────────────
    source_urls = seller_rows["source_url"].dropna().unique()
    if len(source_urls) > 0:
        company["website"] = source_urls[0]

    # ── Sort rows by source trust for field extraction ─────────────────────────
    rows_sorted = seller_rows.copy()
    rows_sorted["_trust_rank"] = rows_sorted["_page_source_type"].apply(source_trust_rank)
    rows_sorted = rows_sorted.sort_values("_trust_rank")

    all_emails, all_phones, all_texts = [], [], []
    all_certs = set()
    sources_used = []

    for _, row in rows_sorted.iterrows():
        blob = safe_str(row.get("_text_blob", ""))
        src  = safe_str(row.get("_page_source_type", "unknown"))
        if not blob:
            continue
        sources_used.append(src)
        all_texts.append(blob)
        all_emails.extend(safe_str(row.get("_emails_found", "")).replace("[", "").replace("]", "").replace("'", "").split(", "))
        all_phones.extend(safe_str(row.get("_phones_found", "")).replace("[", "").replace("]", "").replace("'", "").split(", "))
        all_certs.update(extract_certifications(blob))

    all_emails = [e for e in all_emails if "@" in e]
    all_phones = [p for p in all_phones if len(p) >= 9]

    # ── Company name (conservative: prefer source_url domain) ─────────────────
    website = company["website"]
    if website:
        domain = re.sub(r"https?://(www\.)?", "", website).split("/")[0]
        # Convert domain to probable brand (lotusplywood.com → Lotus Plywood)
        name_parts = re.split(r"[.\-]", domain)[0]
        company["brand_name"] = name_parts.title() if name_parts else ""

    # Try to find explicit company name from high-trust rows
    for _, row in rows_sorted.head(5).iterrows():
        blob = safe_str(row.get("_text_blob", ""))
        # Pattern: "Welcome to X" / "X Pvt Ltd" / "X Private Limited"
        for pat in [
            r"Welcome to ([A-Z][\w\s&.]+(?:Pvt\.?\s*Ltd\.?|Private Limited|Limited|LLP|Inc\.?|LLC)?)",
            r"([A-Z][\w\s&.]+(?:Pvt\.?\s*Ltd\.?|Private Limited|Limited|LLP))",
        ]:
            m = re.search(pat, blob)
            if m:
                candidate = m.group(1).strip()
                if 5 < len(candidate) < 80:
                    company["registered_company_name"] = candidate
                    break
        if company["registered_company_name"]:
            break

    # ── Contact fields ─────────────────────────────────────────────────────────
    if all_emails:
        # Filter out garbled emails (from scraped rows with masked data like in**)
        clean_emails = [e for e in all_emails if not re.search(r"\*", e)]
        company["primary_email"]    = clean_emails[0] if clean_emails else ""
        company["alternate_emails"] = "; ".join(clean_emails[1:4]) if len(clean_emails) > 1 else ""

    if all_phones:
        company["primary_phone"]    = all_phones[0] if all_phones else ""
        company["alternate_phones"] = "; ".join(all_phones[1:4]) if len(all_phones) > 1 else ""

    # ── Address extraction ────────────────────────────────────────────────────
    full_text = " ".join(all_texts)
    # Simple address heuristic: find text near a pincode
    addr_match = re.search(r"([\w\s,.-]{10,60})\s*(\d{6})\b", full_text)
    if addr_match:
        company["registered_address"] = addr_match.group(0).strip()
        company["pincode"]            = addr_match.group(2)

    # ── Business type flags ────────────────────────────────────────────────────
    flags = infer_business_type_flags(full_text)
    for btype, flag in flags.items():
        if flag:
            company["business_type"] = btype
            break  # Use first match
    company["manufacturer_flag"] = flags.get("manufacturer", False)
    company["exporter_flag"]     = flags.get("exporter", False)
    company["importer_flag"]     = flags.get("importer", False)
    company["distributor_flag"]  = flags.get("distributor", False)

    # ── Industry ───────────────────────────────────────────────────────────────
    industry_map = {
        "restaurant": "Food & Beverage", "real_estate": "Real Estate",
        "financial": "Financial Services", "manufacturer": "Manufacturing",
        "trader": "Trading", "retailer": "Retail",
    }
    company["industry"] = industry_map.get(company["business_type"], "")

    # ── Year established ───────────────────────────────────────────────────────
    year = extract_year_established(full_text)
    if year:
        company["year_established"] = year
        try:
            company["years_in_business"] = str(datetime.now().year - int(year))
        except:
            pass

    # ── Certifications ─────────────────────────────────────────────────────────
    company["certifications"] = "; ".join(sorted(all_certs))

    # ── Company description (conservative: from about/homepage only) ───────────
    desc_rows = rows_sorted[rows_sorted["_page_source_type"].isin(["about", "homepage", "generic_page"])]
    # Avoid product rows contaminating description
    desc_rows = desc_rows[~desc_rows["artifact_class"].isin(["ACTUAL_PRODUCT_CANDIDATE"])]
    if not desc_rows.empty:
        best_blob = desc_rows.iloc[0]["_text_blob"]
        # Clean up: remove emails, phones, URLs
        clean_desc = re.sub(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}", "", best_blob)
        clean_desc = re.sub(r"https?://\S+", "", clean_desc)
        clean_desc = re.sub(r"\s+", " ", clean_desc).strip()
        company["company_description"] = clean_desc[:500]

    # ── Social URLs ───────────────────────────────────────────────────────────
    for _, row in rows_sorted.iterrows():
        link = safe_str(row.get("artifact_link", ""))
        for platform, key in [("facebook.com","facebook_url"),("instagram.com","instagram_url"),
                               ("linkedin.com","linkedin_url"),("youtube.com","youtube_url")]:
            if platform in link and not company[key]:
                company[key] = link

    # ── Confidence scoring ─────────────────────────────────────────────────────
    filled = sum(1 for k, v in company.items()
                 if k not in ("seller_id", "country", "company_data_confidence_score",
                              "company_data_sources", "missing_fields", "notes_for_manual_review")
                 and v and v != False)
    total_fields = 20
    company["company_data_confidence_score"] = round(min(filled / total_fields, 1.0), 2)
    company["company_data_sources"] = "; ".join(sorted(set(sources_used)))

    # ── Missing fields ─────────────────────────────────────────────────────────
    critical = ["registered_company_name", "primary_email", "primary_phone", "registered_address", "company_description"]
    missing  = [f for f in critical if not company[f]]
    company["missing_fields"] = "; ".join(missing)
    if missing:
        company["notes_for_manual_review"] = f"Missing: {'; '.join(missing)}"

    return company


log.info("✅ Company extraction functions loaded.")


## Cell 12 — Product Validation (Most Important)

In [ ]:
# ── Scoring weights ───────────────────────────────────────────────────────────
PRODUCT_SIGNAL_WEIGHTS = {
    "has_sku":                  0.25,
    "has_price":                0.15,
    "has_specs":                0.20,
    "has_specific_description": 0.15,
    "has_product_url":          0.10,
    "has_product_image":        0.05,
    "name_specificity":         0.15,
    "image_cls_boost":          0.05,   # Secondary hook modifier only
}

GENERIC_NAME_PATTERNS = [
    r"^(our|the|a|an)\s+",
    r"^\d+[+\-]?\s*(year|client|project|sq|sqft|happy|customer|employee)",
    r"^(why us|why choose|how we|what we|who we)",
    r"^(vision|mission|values?|philosophy|approach|strategy|culture)",
    r"^(news|blog|press|media|event|award|certificate|achievement)",
    r"^(home|homepage|index|main|landing)\s*$",
]

STRONG_PRODUCT_NAME_SIGNALS = [
    r"\b(kit|set|pack|unit|piece|pcs|box|bag|roll|sheet|plate|tube|bar|rod|wire)\b",
    r"\b(model|series|type|grade|variant|edition|version|gen)\s*[A-Z0-9\-]+",
    r"\b\d+\s*(mm|cm|m|kg|g|ml|l|w|kw|hp|amp|v|volt|rpm|psi|bar|mpa)\b",
    r"\b(grade|alloy|class|category)\s+[A-Z0-9\-]+",
]


def score_product_candidate(row: pd.Series) -> dict:
    """
    Multi-signal product validation scoring.
    Returns score breakdown and final score.
    """
    name    = safe_str(row.get("item_name", ""))
    desc    = safe_str(row.get("description", ""))
    specs   = safe_str(row.get("product_specifications", ""))
    sku     = safe_str(row.get("sku", ""))
    price   = safe_str(row.get("price", ""))
    art_lnk = safe_str(row.get("artifact_link", ""))
    img_url = safe_str(row.get("image_url", ""))
    img_cls = safe_str(row.get("_image_cls_result", ""))
    img_cls_conf = float(row.get("_image_cls_conf", 0.0) or 0.0)
    blob    = safe_str(row.get("_text_blob", ""))

    signals = {}

    # ── SKU / model number ────────────────────────────────────────────────────
    signals["has_sku"] = bool(sku and len(sku) >= 3 and not sku.lower().startswith("in"))

    # ── Price ─────────────────────────────────────────────────────────────────
    signals["has_price"] = bool(price and re.search(r"\d", price))

    # ── Specs ─────────────────────────────────────────────────────────────────
    signals["has_specs"] = bool(specs and len(specs) > 20)

    # ── Description quality ───────────────────────────────────────────────────
    desc_len = len(desc)
    has_numeric_in_desc = bool(re.search(r"\d+\s*(mm|cm|kg|g|ml|l|w|kw|hp|v|rpm|psi)", desc, re.I))
    signals["has_specific_description"] = bool(desc_len > 50 and (has_numeric_in_desc or desc_len > 150))

    # ── Product URL ───────────────────────────────────────────────────────────
    signals["has_product_url"] = is_likely_product_url(art_lnk) if art_lnk else False

    # ── Image ─────────────────────────────────────────────────────────────────
    signals["has_product_image"] = bool(img_url and not any(
        k in img_url.lower() for k in ["logo", "banner", "icon", "favicon", "bg", "background"]
    ))

    # ── Name specificity ──────────────────────────────────────────────────────
    name_score = 0.0
    if name and len(name) > 3:
        # Penalize generic names
        if not matches_any_pattern(name, GENERIC_NAME_PATTERNS):
            name_score += 0.5
        # Boost for specific product signals in name
        if any(re.search(p, name, re.I) for p in STRONG_PRODUCT_NAME_SIGNALS):
            name_score += 0.5
        # Penalize very long names (often marketing text, not product names)
        if len(name) > 120:
            name_score -= 0.3
        # Penalize if name == description (scraped heading)
        if name.lower().strip() == desc.lower().strip():
            name_score -= 0.2
    signals["name_specificity"] = max(0.0, min(1.0, name_score))

    # ── Image classifier as secondary modifier ─────────────────────────────────
    if img_cls == "product" and img_cls_conf > 0.6:
        signals["image_cls_boost"] = 0.5  # Partial boost, not full
    else:
        signals["image_cls_boost"] = 0.0

    # ── Weighted score ────────────────────────────────────────────────────────
    total = sum(PRODUCT_SIGNAL_WEIGHTS[k] * (1.0 if isinstance(v, bool) and v else float(v) if not isinstance(v, bool) else 0.0)
                for k, v in signals.items())
    return {"signals": signals, "score": round(total, 3)}


def hard_exclusion_check(row: pd.Series) -> tuple[bool, str]:
    """
    Returns (should_exclude: bool, reason: str).
    Strong non-product archetypes are blocked regardless of score.
    Override only if there is very strong contrary evidence (SKU + price + specs).
    """
    name = safe_str(row.get("item_name", ""))
    desc = safe_str(row.get("description", ""))
    blob = safe_str(row.get("_text_blob", ""))

    if matches_any_pattern(name, HARD_EXCLUSION_PATTERNS):
        # Check for strong override evidence
        sku   = safe_str(row.get("sku", ""))
        price = safe_str(row.get("price", ""))
        specs = safe_str(row.get("product_specifications", ""))
        if sku and price and specs:
            return False, "hard_exclusion_overridden_by_sku_price_specs"
        return True, f"hard_exclusion_name_matches_non_product_archetype"

    if matches_any_pattern(desc, HARD_EXCLUSION_PATTERNS) and not safe_str(row.get("sku", "")):
        return True, "hard_exclusion_desc_matches_non_product_archetype"

    # Category vs product check
    if matches_any_pattern(name, CATEGORY_PATTERNS):
        sku = safe_str(row.get("sku", ""))
        return (True, "category_level_entry") if not sku else (False, "category_overridden_by_sku")

    return False, ""


def classify_entity_level(row: pd.Series) -> str:
    """Determine if this is PRODUCT / CATEGORY / UNKNOWN level."""
    name = safe_str(row.get("item_name", ""))
    if matches_any_pattern(name, CATEGORY_PATTERNS):
        return "CATEGORY"
    if safe_str(row.get("sku", "")) or safe_str(row.get("price", "")):
        return "PRODUCT"
    score_result = score_product_candidate(row)
    if score_result["score"] >= 0.4:
        return "PRODUCT"
    return "UNKNOWN"


def validate_products(df: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    """
    Apply product validation to all ACTUAL_PRODUCT_CANDIDATE rows.
    Also selectively invokes LLM hook for borderline cases.
    Returns validated DataFrame and hook log entries.
    """
    df = df.copy()
    df["is_actual_product"]          = False
    df["product_confidence_score"]   = 0.0
    df["product_validation_reasoning"] = ""
    df["hard_excluded"]              = False
    df["hard_exclusion_reason"]      = ""
    df["entity_level"]               = "UNKNOWN"
    df["needs_manual_review"]        = False

    hook_log = []
    llm_count = 0

    candidate_mask = df["artifact_class"] == "ACTUAL_PRODUCT_CANDIDATE"

    for idx, row in df[candidate_mask].iterrows():
        # ── Hard exclusion first ───────────────────────────────────────────────
        excluded, excl_reason = hard_exclusion_check(row)
        if excluded:
            df.at[idx, "hard_excluded"]         = True
            df.at[idx, "hard_exclusion_reason"] = excl_reason
            df.at[idx, "product_validation_reasoning"] = f"EXCLUDED: {excl_reason}"
            continue

        # ── Score ─────────────────────────────────────────────────────────────
        score_result = score_product_candidate(row)
        score   = score_result["score"]
        signals = score_result["signals"]

        # ── LLM hook for borderline cases ─────────────────────────────────────
        if (ENABLE_LLM_EXTRACTION_HOOK and
            llm_count < LLM_MAX_ROWS_PER_RUN and
            0.20 <= score <= LLM_MIN_CONFIDENCE_TO_TRIGGER):
            text = safe_str(row.get("_text_blob", ""))
            llm_result = hook_llm_extraction(text, task="product")
            if not llm_result.get("skipped"):
                # If LLM found a product name that differs from item_name, boost slightly
                if llm_result.get("product_name") and len(safe_str(llm_result.get("product_name",""))) > 5:
                    score = min(score + 0.08, 1.0)
                df.at[idx, "_llm_extracted"] = json.dumps(llm_result)
                triggered = df.at[idx, "_hook_triggered"]
                df.at[idx, "_hook_triggered"] = (triggered + "|LLM").strip("|")
                hook_log.append({
                    "row_idx": idx, "hook": "LLM",
                    "url": row.get("artifact_link", ""),
                    "result_preview": str(llm_result)[:150],
                    "method": "flan-t5-base",
                    "confidence": score,
                })
                llm_count += 1

        # ── Entity level ───────────────────────────────────────────────────────
        entity_level = classify_entity_level(row)
        df.at[idx, "entity_level"]              = entity_level
        df.at[idx, "product_confidence_score"]  = score

        if entity_level == "CATEGORY":
            df.at[idx, "product_validation_reasoning"] = f"CATEGORY (score={score:.2f}): not product-level"
            df.at[idx, "needs_manual_review"] = score >= 0.35  # Borderline categories
            continue

        # ── Final inclusion decision ───────────────────────────────────────────
        is_product = score >= PRODUCT_CONFIDENCE_THRESHOLD
        df.at[idx, "is_actual_product"] = is_product
        df.at[idx, "needs_manual_review"] = (
            PRODUCT_CONFIDENCE_THRESHOLD - 0.10 <= score < PRODUCT_CONFIDENCE_THRESHOLD
        )

        signal_str = "; ".join([f"{k}={v}" for k, v in signals.items()])
        df.at[idx, "product_validation_reasoning"] = (
            f"score={score:.2f} | {'ACCEPTED' if is_product else 'REJECTED'} | {signal_str}"
        )

    log.info(f"Product validation: "
             f"{df['is_actual_product'].sum()} accepted | "
             f"{df['hard_excluded'].sum()} hard excluded | "
             f"{df['needs_manual_review'].sum()} need review")
    return df, hook_log


log.info("✅ Product validation loaded.")


## Cell 13 — Product Catalogue Extraction

In [ ]:
def build_product_catalogue(df: pd.DataFrame) -> pd.DataFrame:
    """Build PRODUCT_CATALOGUE from rows where is_actual_product == True."""
    product_rows = df[df["is_actual_product"] == True].copy()
    if product_rows.empty:
        log.warning("No products accepted! Check thresholds or input data.")
        return pd.DataFrame()

    records = []
    for i, (_, row) in enumerate(product_rows.iterrows()):
        name  = safe_str(row.get("item_name", ""))
        desc  = safe_str(row.get("description", ""))
        specs = safe_str(row.get("product_specifications", ""))
        sku   = safe_str(row.get("sku", ""))

        # ── Spec fetch hook ────────────────────────────────────────────────────
        fetched_specs = ""
        fetch_method  = "not_attempted"
        if not specs and row.get("artifact_link") and i < MAX_SPEC_FETCH_PER_RUN:
            fetch_result  = hook_fetch_product_specs(safe_str(row.get("artifact_link", "")))
            fetched_specs = fetch_result.get("specs_text", "")
            fetch_method  = fetch_result.get("method", "")

        final_specs = specs or fetched_specs

        # ── Extract spec fields ────────────────────────────────────────────────
        dims  = re.findall(r"\d+(?:\.\d+)?\s*(?:mm|cm|m|inch|ft)", final_specs + " " + desc, re.I)
        power = re.findall(r"\d+(?:\.\d+)?\s*(?:w|kw|hp|watt|kilowatt)", final_specs + " " + desc, re.I)
        cap   = re.findall(r"\d+(?:\.\d+)?\s*(?:l|ltr|litre|ml|kg|ton|mt|kl)", final_specs + " " + desc, re.I)
        mods  = re.findall(r"(?:model|series|type)\s*[:\-]?\s*([A-Z0-9\-]+)", final_specs + " " + name, re.I)
        mats  = re.findall(r"(?:stainless steel|mild steel|aluminum|aluminium|brass|copper|plastic|"
                           r"PVC|PP|HDPE|LDPE|cast iron|wrought iron|galvanized|teak|pine|oak|marble|granite)", 
                           final_specs + " " + desc, re.I)

        records.append({
            "seller_id":                safe_str(row.get("seller_id", "")),
            "product_id":               f"P{i+1:04d}",
            "product_name":             name,
            "normalized_product_name":  name.lower().strip(),
            "product_category":         safe_str(row.get("category_format", "")),
            "sub_category":             "",
            "entity_level":             safe_str(row.get("entity_level", "PRODUCT")),
            "brand_if_any":             safe_str(row.get("brand", "")),
            "sku_if_any":               sku,
            "short_description":        desc[:200] if desc else "",
            "long_description":         desc,
            "key_features":             "",  # Could be LLM-extracted if hook enabled
            "applications":             "",
            "materials_if_any":         "; ".join(set(mats)) if mats else "",
            "specifications_extracted": final_specs[:1000] if final_specs else "Not Acquired",
            "specifications_available_flag": "Yes" if final_specs else "No",
            "specification_table_json": "",
            "dimensions_if_any":        "; ".join(set(dims)) if dims else "",
            "capacity_if_any":          "; ".join(set(cap)) if cap else "",
            "power_if_any":             "; ".join(set(power)) if power else "",
            "model_numbers_if_any":     "; ".join(set(mods)) if mods else "",
            "product_image_urls":       safe_str(row.get("image_url", "")),
            "product_page_url":         safe_str(row.get("artifact_link", "")),
            "brochure_reference":       safe_str(row.get("document_url", "")),
            "source_text_snippet":      safe_str(row.get("_text_blob", ""))[:300],
            "is_actual_product":        True,
            "product_confidence_score": safe_str(row.get("product_confidence_score", "")),
            "product_validation_reasoning": safe_str(row.get("product_validation_reasoning", "")),
            "duplicate_group":          "",  # Filled in dedup step
            "needs_manual_review":      safe_str(row.get("needs_manual_review", "")),
            "notes_for_manual_review":  "",
            "_spec_fetch_method":       fetch_method,
        })

    catalogue = pd.DataFrame(records)
    log.info(f"Product catalogue: {len(catalogue)} rows")
    return catalogue


log.info("✅ Product extraction loaded.")


## Cell 14 — Seller Offerings Extraction

In [ ]:
def build_seller_offerings(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract commercially useful non-product offerings:
    categories, families, collections, menu sections, solution buckets.
    These do NOT meet product-level criteria but are useful for seller enrichment.
    """
    # Rows that are category-level or brochure-sourced but not actual products
    offering_mask = (
        (df["entity_level"] == "CATEGORY") |
        (df["artifact_class"] == "DOCUMENT_OR_BROCHURE") |
        (
            (df["artifact_class"] == "ACTUAL_PRODUCT_CANDIDATE") &
            (df["is_actual_product"] == False) &
            (df["hard_excluded"] == False) &
            (df["product_confidence_score"].astype(float) >= 0.15)
        )
    )
    offering_rows = df[offering_mask].copy()

    records = []
    for i, (_, row) in enumerate(offering_rows.iterrows()):
        name = safe_str(row.get("item_name", "")) or safe_str(row.get("description", ""))[:60]
        if not name or len(name) < 3:
            continue

        offering_type = "category"
        if row.get("artifact_class") == "DOCUMENT_OR_BROCHURE":
            offering_type = "brochure"
        elif "menu" in name.lower() or "food" in safe_str(row.get("source_url","")).lower():
            offering_type = "menu_section"
        elif row.get("entity_level") == "CATEGORY":
            offering_type = "product_family"

        records.append({
            "seller_id":             safe_str(row.get("seller_id", "")),
            "offering_id":           f"O{i+1:04d}",
            "offering_name":         name[:200],
            "offering_type":         offering_type,
            "normalized_offering_name": name.lower().strip()[:200],
            "source_row":            safe_str(row.get("_raw_row_idx", "")),
            "source_url":            safe_str(row.get("found_on_page", "")),
            "source_text_snippet":   safe_str(row.get("_text_blob", ""))[:300],
            "confidence_score":      safe_str(row.get("product_confidence_score", 0.0)),
            "notes_for_manual_review": safe_str(row.get("product_validation_reasoning", ""))[:200],
        })

    offerings = pd.DataFrame(records) if records else pd.DataFrame()
    log.info(f"Seller offerings: {len(offerings)} rows")
    return offerings


log.info("✅ Seller offerings extraction loaded.")


## Cell 15 — Deduplication

In [ ]:
def deduplicate_products(catalogue: pd.DataFrame) -> pd.DataFrame:
    """
    Deduplicate product rows using:
    - Normalized name fuzzy similarity
    - Same artifact_link URL
    - Same SKU
    Groups duplicates and assigns duplicate_group ID.
    """
    if catalogue.empty:
        return catalogue

    cat = catalogue.copy()
    cat["duplicate_group"] = ""
    n = len(cat)
    group_id = 1
    assigned = {}  # idx -> group_id

    names = cat["normalized_product_name"].tolist()
    urls  = cat["product_page_url"].tolist()
    skus  = cat["sku_if_any"].tolist()

    for i in range(n):
        if i in assigned:
            continue
        assigned[i] = group_id
        for j in range(i + 1, n):
            if j in assigned:
                continue
            # Check merge conditions
            name_sim  = fuzz.token_sort_ratio(names[i], names[j]) if names[i] and names[j] else 0
            same_url  = urls[i] and urls[j] and urls[i] == urls[j]
            same_sku  = skus[i] and skus[j] and skus[i].lower() == skus[j].lower()
            if name_sim >= DEDUP_SIMILARITY_THRESHOLD or same_url or same_sku:
                assigned[j] = group_id
        group_id += 1

    for idx, gid in assigned.items():
        cat.at[cat.index[idx], "duplicate_group"] = f"G{gid:04d}"

    # Mark duplicates: keep first in each group
    cat["_is_duplicate"] = cat.duplicated(subset=["duplicate_group"], keep="first")
    dupes = cat["_is_duplicate"].sum()
    log.info(f"Deduplication: {n} → {n - dupes} unique products ({dupes} duplicates)")

    # Remove exact duplicates (keep first)
    cat_deduped = cat[~cat["_is_duplicate"]].drop(columns=["_is_duplicate"]).reset_index(drop=True)
    return cat_deduped


log.info("✅ Deduplication loaded.")


## Cell 16 — Debug / Intermediate Outputs

In [ ]:
def save_debug_outputs(clean_df: pd.DataFrame, enriched_df: pd.DataFrame,
                       validated_df: pd.DataFrame, hook_log: list) -> None:
    """Save all intermediate debug files."""

    # ── Cleaned intermediate data ─────────────────────────────────────────────
    debug_cols = [c for c in clean_df.columns if not c.startswith("_raw_")]
    safe_excel_write(clean_df[debug_cols], str(OUTPUT_DIR / "cleaned_intermediate_data.xlsx"))

    # ── Artifact classification output ─────────────────────────────────────────
    cls_cols = ["seller_id", "source_url", "found_on_page", "item_type", "item_name",
                "description", "artifact_class", "_page_source_type", "_emails_found",
                "_phones_found", "_raw_row_idx"]
    cls_cols = [c for c in cls_cols if c in enriched_df.columns]
    safe_excel_write(enriched_df[cls_cols], str(OUTPUT_DIR / "artifact_classification_output.xlsx"))

    # ── Product validation debug ───────────────────────────────────────────────
    val_cols = ["seller_id", "item_name", "description", "sku", "price", "artifact_class",
                "entity_level", "is_actual_product", "product_confidence_score",
                "product_validation_reasoning", "hard_excluded", "hard_exclusion_reason",
                "needs_manual_review", "_hook_triggered", "_image_cls_result",
                "_image_cls_conf", "_raw_row_idx"]
    val_cols = [c for c in val_cols if c in validated_df.columns]
    safe_excel_write(validated_df[val_cols], str(OUTPUT_DIR / "product_validation_debug_output.xlsx"))

    # ── Hook debug log ─────────────────────────────────────────────────────────
    if hook_log:
        hook_df = pd.DataFrame(hook_log)
        safe_excel_write(hook_df, str(OUTPUT_DIR / "hook_debug_log_output.xlsx"))
    else:
        log.info("No hook invocations to log.")


log.info("✅ Debug output functions loaded.")


## Cell 17 — Export Primary Outputs

In [ ]:
def save_primary_outputs(company_df: pd.DataFrame,
                         catalogue_df: pd.DataFrame,
                         offerings_df: pd.DataFrame) -> None:
    """Save primary output Excel files."""
    safe_excel_write(company_df,  str(OUTPUT_DIR / "company_details_output.xlsx"))
    safe_excel_write(catalogue_df, str(OUTPUT_DIR / "product_catalogue_output.xlsx"))
    if not offerings_df.empty:
        safe_excel_write(offerings_df, str(OUTPUT_DIR / "seller_offerings_output.xlsx"))

    # Processing summary
    summary = {
        "run_id":         RUN_ID,
        "n_companies":    len(company_df),
        "n_products":     len(catalogue_df),
        "n_offerings":    len(offerings_df) if not offerings_df.empty else 0,
        "timestamp":      datetime.now().isoformat(),
    }
    with open(OUTPUT_DIR / "processing_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    log.info(f"Processing summary: {summary}")


log.info("✅ Export functions loaded.")


## Cell 18 — Full Pipeline Runner

In [ ]:
def run_pipeline(file_path: str) -> dict:
    """
    End-to-end pipeline runner.
    Returns dict with all output DataFrames.
    """
    log.info(f"{'='*60}")
    log.info(f"PIPELINE START | file={file_path} | run={RUN_ID}")
    log.info(f"{'='*60}")
    t0 = time.time()

    # ── Step 1: Ingest ────────────────────────────────────────────────────────
    log.info("[Step 1] Data Ingestion")
    raw_df   = load_input_file(file_path)

    # ── Step 2: Schema Inference ──────────────────────────────────────────────
    log.info("[Step 2] Schema Inference")
    role_map = infer_column_roles(raw_df)

    # ── Step 3: Clean & Normalize ──────────────────────────────────────────────
    log.info("[Step 3] Cleaning & Normalization")
    clean_df = clean_and_normalize(raw_df, role_map)

    # ── Step 4: Artifact Classification ───────────────────────────────────────
    log.info("[Step 4] Artifact Classification")
    classified_df = add_artifact_classification(clean_df)

    # ── Step 5: Enrichment Hooks (OCR, Image Cls) ─────────────────────────────
    log.info("[Step 5] Artifact Enrichment (Hooks 1 & 2)")
    enriched_df, hook_log_enrich = enrich_artifacts(classified_df)

    # ── Step 6: Company Extraction ────────────────────────────────────────────
    log.info("[Step 6] Company Details Extraction")
    company_records = []
    for seller_id, seller_rows in enriched_df.groupby("seller_id"):
        rec = extract_company_for_seller(seller_rows)
        company_records.append(rec)
    company_df = pd.DataFrame(company_records)
    log.info(f"  Extracted {len(company_df)} company records")

    # ── Step 7: Product Validation (Hooks 3) ──────────────────────────────────
    log.info("[Step 7] Product Validation")
    validated_df, hook_log_validate = validate_products(enriched_df)

    # ── Step 8: Product Catalogue ─────────────────────────────────────────────
    log.info("[Step 8] Product Catalogue Extraction (Hook 4)")
    catalogue_df = build_product_catalogue(validated_df)

    # ── Step 9: Seller Offerings ──────────────────────────────────────────────
    log.info("[Step 9] Seller Offerings Extraction")
    offerings_df = build_seller_offerings(validated_df)

    # ── Step 10: Deduplication ────────────────────────────────────────────────
    log.info("[Step 10] Deduplication")
    if not catalogue_df.empty:
        catalogue_df = deduplicate_products(catalogue_df)

    # ── Step 11: Save Debug Outputs ───────────────────────────────────────────
    log.info("[Step 11] Saving Debug Outputs")
    all_hook_logs = hook_log_enrich + hook_log_validate
    save_debug_outputs(clean_df, enriched_df, validated_df, all_hook_logs)

    # ── Step 12: Save Primary Outputs ─────────────────────────────────────────
    log.info("[Step 12] Saving Primary Outputs")
    save_primary_outputs(company_df, catalogue_df, offerings_df)

    elapsed = round(time.time() - t0, 1)
    log.info(f"{'='*60}")
    log.info(f"PIPELINE COMPLETE | {elapsed}s | outputs → {OUTPUT_DIR}/")
    log.info(f"{'='*60}")

    return {
        "company_df":   company_df,
        "catalogue_df": catalogue_df,
        "offerings_df": offerings_df,
        "validated_df": validated_df,
        "enriched_df":  enriched_df,
    }


log.info("✅ Pipeline runner loaded.")


## Cell 19 — Upload File & Run Pipeline
Run this cell to upload your Excel/CSV file and execute the full pipeline.  
If running in Colab, a file-picker dialog will appear. Otherwise, set `INPUT_FILE` directly.

In [ ]:
import os

# ── Option A: Colab file upload ────────────────────────────────────────────────
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

INPUT_FILE = None  # Set this manually if not in Colab, e.g. "my_sellers.xlsx"

if IN_COLAB and INPUT_FILE is None:
    print("📂 Please upload your Excel or CSV file...")
    uploaded = colab_files.upload()
    if uploaded:
        INPUT_FILE = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {INPUT_FILE}")
    else:
        raise ValueError("No file uploaded. Please upload a file and re-run.")

if INPUT_FILE is None:
    raise ValueError("Set INPUT_FILE to the path of your Excel/CSV file.")

# ── Run pipeline ───────────────────────────────────────────────────────────────
results = run_pipeline(INPUT_FILE)

# ── Quick summary ──────────────────────────────────────────────────────────────
print("\n" + "="*50)
print("PIPELINE OUTPUT SUMMARY")
print("="*50)
print(f"  Companies extracted : {len(results['company_df'])}")
print(f"  Products accepted   : {len(results['catalogue_df'])}")
print(f"  Seller offerings    : {len(results['offerings_df'])}")
print(f"  Output files in     : {OUTPUT_DIR}/")
print("="*50)

# Show product table preview
if not results["catalogue_df"].empty:
    preview_cols = ["seller_id", "product_name", "sku_if_any", "product_confidence_score",
                    "specifications_available_flag", "needs_manual_review"]
    preview_cols = [c for c in preview_cols if c in results["catalogue_df"].columns]
    print("\n📦 Product Catalogue Preview:")
    display(results["catalogue_df"][preview_cols].head(10))

if not results["company_df"].empty:
    print("\n🏢 Company Details Preview:")
    preview_company = ["seller_id","registered_company_name","brand_name","website",
                       "primary_phone","primary_email","industry","company_data_confidence_score"]
    preview_company = [c for c in preview_company if c in results["company_df"].columns]
    display(results["company_df"][preview_company].head(10))


## Cell 20 — Download Output Files (Colab)

In [ ]:
# Run this cell to download all output files in Google Colab
try:
    from google.colab import files as colab_files
    output_files = list(OUTPUT_DIR.glob("*.xlsx")) + list(OUTPUT_DIR.glob("*.json"))
    for f in sorted(output_files):
        print(f"Downloading: {f.name}")
        colab_files.download(str(f))
except ImportError:
    print(f"Not in Colab. Files are available in: {OUTPUT_DIR}/")
    for f in sorted(OUTPUT_DIR.glob("*")):
        print(f"  {f.name}")


## Cell 21 — Optional Evaluation (Hook 5)
If you have manually labeled ground truth, provide it here to evaluate pipeline accuracy.

In [ ]:
# ── Optional: provide a ground truth file ─────────────────────────────────────
# Ground truth file must have columns:
#   _raw_row_idx  (matches pipeline output)
#   is_actual_product  (0 or 1)

GROUND_TRUTH_FILE = None  # Set to path of ground truth xlsx, e.g. "ground_truth.xlsx"
ENABLE_EVALUATION_HOOK = True if GROUND_TRUTH_FILE else False

if ENABLE_EVALUATION_HOOK and GROUND_TRUTH_FILE:
    gt_df = pd.read_excel(GROUND_TRUTH_FILE, dtype=str)
    gt_df["_raw_row_idx"]     = gt_df["_raw_row_idx"].astype(int)
    gt_df["is_actual_product"] = gt_df["is_actual_product"].astype(int)

    eval_metrics = hook_evaluate_precision_recall(
        predicted_df=results["validated_df"],
        ground_truth_df=gt_df,
    )
    print("\n📊 Evaluation Metrics:")
    for k, v in eval_metrics.items():
        print(f"  {k}: {v}")

    # Save metrics
    with open(OUTPUT_DIR / "evaluation_metrics_output.json", "w") as f:
        json.dump(eval_metrics, f, indent=2)
    print(f"\nSaved → {OUTPUT_DIR}/evaluation_metrics_output.json")
else:
    print("Evaluation skipped. Set GROUND_TRUTH_FILE to enable.")


## Cell 22 — Tuning Guide & Notes

### Tuning Thresholds
| Parameter | Default | Effect |
|-----------|---------|--------|
| `PRODUCT_CONFIDENCE_THRESHOLD` | 0.40 | Lower → more products (more false positives). Higher → fewer products (more false negatives). |
| `DEDUP_SIMILARITY_THRESHOLD` | 88 | Lower → more aggressive dedup. |
| `LLM_MIN_CONFIDENCE_TO_TRIGGER` | 0.35 | Lower → LLM called more often. |

### Enabling Hooks
1. **Hook 1 (OCR):** `ENABLE_OCR_HOOK = True` + uncomment `!pip install easyocr`
2. **Hook 2 (Image Cls):** `ENABLE_IMAGE_CLASSIFIER_HOOK = True` + uncomment `!pip install open-clip-torch`
3. **Hook 3 (LLM):** `ENABLE_LLM_EXTRACTION_HOOK = True` + uncomment `!pip install transformers accelerate`
4. **Hook 4 (Spec Fetch):** `ENABLE_SPEC_FETCH_HOOK = True` (beautifulsoup4 already installed)
5. **Hook 5 (Evaluation):** Provide `GROUND_TRUTH_FILE`

### Swapping Models
- **OCR:** Replace EasyOCR with `doctr` for better layout understanding. Install: `!pip install python-doctr[torch]`
- **Image Cls:** Upgrade to `SigLIP` (ViT-SO400M) for better zero-shot accuracy. Heavier but free.
- **LLM:** Upgrade to `mistralai/Mistral-7B-Instruct-v0.3` (needs Colab Pro A100) or `meta-llama/Llama-3.2-3B-Instruct` (lighter).

### Colab Compute Notes
- All hooks are **disabled by default** — the pipeline runs fully on CPU in ~5–30 seconds for 100–1000 rows.
- Enabling OCR adds ~30s first-run model download + ~1–3s per image.
- Enabling image classification adds ~1–2 min model download + ~0.2–0.5s per image on CPU.
- Enabling LLM adds ~1 min model download + ~2–5s per LLM call on CPU (flan-t5-base).
- Use `LLM_MAX_ROWS_PER_RUN` and `MAX_SPEC_FETCH_PER_RUN` to cap time.

### Adding New Column Patterns
Edit `SEMANTIC_ROLE_KEYWORDS` in Cell 3 to add new column aliases for your dataset.

### Adding Hard Exclusions
Add new regex patterns to `HARD_EXCLUSION_PATTERNS` in Cell 3 for domain-specific non-product archetypes.
